# Returns-Mode Training & Portfolio Backtesting (4-Hourly BTC)

This notebook runs the **returns-mode** hierarchical QLSTM and the **vectorbt portfolio backtest** end-to-end on Google Colab.

It covers:
1. Train a hierarchical QLSTM with `target_mode: return` (predict returns, reconstruct prices) — runs in the main env.
2. Export a **per-bar predictions CSV** with `decision_time` / `target_time` (Oracle O1 alignment).
3. Run the **alignment gate** (proves no off-by-one / no leakage).
4. Backtest a **single coin** with vectorbt — next-bar-open execution, fees, slippage — in an **isolated env**.
5. Backtest a **multi-coin basket** (pooled NAV via `cash_sharing`).
6. Render equity / drawdown plots.

---

## Why two environments?

vectorbt depends on numba and requires `numpy < 2` / `pandas < 3`, which is **incompatible** with the PyTorch training stack (numpy 2.x). So:

| Step | Environment | Why |
|------|-------------|-----|
| Train + export predictions CSV | **Colab default** (numpy 2.x, torch) | model training |
| Backtest (vectorbt) | **isolated venv** (`numpy<2`) | vectorbt/numba |

The training step writes a plain CSV; the backtest step reads that CSV. They never share a Python process, so the version conflict never bites.

---

## Requirements
- GPU runtime recommended (Runtime → Change runtime type → T4 GPU).
- The LunarCrush data cache CSVs are **not** in the repo. Upload them to Google Drive once (see Section 1).

## 1. Setup: clone repo, get data, mount Drive

In [ ]:
# Clone (or update) the thesis repository
import os
if not os.path.exists('/content/thesis'):
    !git clone https://github.com/BerkayClik/thesis.git /content/thesis
%cd /content/thesis
!git pull

In [ ]:
# Mount Google Drive (for the data cache + saving results)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Make the LunarCrush cache CSVs available at data/cache/.
#
# The cache is NOT committed to the repo. Two options:
#
#   (A) Upload the CSVs to Google Drive ONCE, then copy them in here.
#       Put them in a Drive folder and set DRIVE_DATA_DIR below.
#
#   (B) Upload directly from your machine each session (uncomment the
#       files.upload() block).
import os, shutil, glob

os.makedirs('data/cache', exist_ok=True)

# --- Option A: copy from Drive (recommended) ---
DRIVE_DATA_DIR = '/content/drive/MyDrive/thesis_data'   # <-- put your CSVs here
if os.path.isdir(DRIVE_DATA_DIR):
    for src in glob.glob(f'{DRIVE_DATA_DIR}/lunarcrush_*4hour_full.csv'):
        shutil.copy(src, 'data/cache/')
        print('copied', os.path.basename(src))

# --- Option B: manual upload (uncomment if not using Drive) ---
# from google.colab import files
# uploaded = files.upload()  # select lunarcrush_btc_4hour_full.csv (+ others)
# for name in uploaded:
#     shutil.move(name, f'data/cache/{name}')

print('\ndata/cache contents:')
!ls -la data/cache/lunarcrush_*4hour*.csv 2>/dev/null || echo 'No 4-hour CSVs found — upload them first.'

In [ ]:
# Training-side dependencies (main env)
!pip install -q yfinance scipy seaborn

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Make a return-mode config

We take the existing `configs/data/4hourly/btc_hier.yaml` and add a single key: `target_mode: return`. Everything else is unchanged. (Default `target_mode` is `price`, so the original configs keep working untouched.)

In [ ]:
import yaml, os

os.makedirs('configs/data/4hourly', exist_ok=True)

def make_return_config(src_path, coin, out_path, results_dir):
    cfg = yaml.safe_load(open(src_path))
    cfg['data']['coin'] = coin
    cfg['data']['data_path'] = f'data/cache/lunarcrush_{coin}_4hour_full.csv'
    cfg['data']['target_mode'] = 'return'   # the only functional change
    cfg['output']['results_dir'] = results_dir
    cfg['output']['checkpoint_dir'] = f'checkpoints/{coin}_4h_return'
    yaml.safe_dump(cfg, open(out_path, 'w'))
    print(f'wrote {out_path} (coin={coin}, target_mode=return)')

make_return_config('configs/data/4hourly/btc_hier.yaml', 'btc',
                   'configs/data/4hourly/btc_hier_return.yaml',
                   'experiments/results/btc_4h_return')

In [ ]:
# A small experiment config: one hierarchical variant, one seed.
# Bump num_epochs for a real run (e.g. 100). Kept modest here for speed.
exp_yaml = '''
experiment:
  name: "btc_4h_return"
  description: "hierarchical QLSTM, return-mode, for backtesting"
  seeds: [42]
  variants:
    - name: "hier_qlstm_concat"
      model:
        type: "hier_qlstm_concat"
        hidden_size: 32
        num_layers: 2
        dropout: 0.1
training:
  batch_size: 32
  num_epochs: 50
  learning_rate: 0.0005
  patience: 10
  seed: 42
output:
  results_dir: "experiments/results/btc_4h_return"
'''
open('configs/experiments/btc_4h_return.yaml', 'w').write(exp_yaml)
print('wrote configs/experiments/btc_4h_return.yaml')

## 3. Train (return-mode) and export the predictions CSV

The runner trains the model, evaluates on the test split, and writes:
- the usual results JSON (now with `target_mode` + `predictions_csv_path`), and
- a **per-bar predictions CSV**: `decision_time, target_time, prev_close, pred_close, true_close, pred_return, true_return`.

Low MAPE on prices can be misleading — the backtest in Section 5 is the honest judge of whether there's any tradeable edge.

In [ ]:
os.environ['PYTHONPATH'] = '/content/thesis'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

!python experiments/run_experiments.py \
    --base-config configs/data/4hourly/btc_hier_return.yaml \
    --experiment-config configs/experiments/btc_4h_return.yaml

In [ ]:
# Locate the predictions CSV that was written
import glob
btc_csvs = glob.glob('experiments/results/btc_4h_return/*_predictions.csv')
print('predictions CSVs:', btc_csvs)
BTC_PRED_CSV = btc_csvs[0]

import pandas as pd
pd.read_csv(BTC_PRED_CSV).head()

## 4. Alignment gate (Oracle O5)

Before backtesting, prove the predictions CSV is leakage-free: `target_time` is strictly after `decision_time`, the bar gap is constant (4h), and `pred_close == prev_close*(1+pred_return)`. Exit code 0 = safe to backtest.

In [ ]:
!python scripts/verify_alignment.py "{BTC_PRED_CSV}" --rows 5 --constant-gap

## 5. Set up the isolated backtest environment (uv + Python 3.11)

vectorbt needs `numpy<2`, so it lives in its own venv. We use `uv` to build it fast. This runs once per session.

In [ ]:
# Install uv
!pip install -q uv

# Create the isolated backtest venv (Python 3.11) and install pinned vectorbt stack
!uv venv --python 3.11 .venv-backtest
!uv pip install --python .venv-backtest -r requirements-backtest.txt

In [ ]:
# Smoke test: must print 'vbt OK <version>' and a finite Total Return
!.venv-backtest/bin/python scripts/backtest_env_smoke.py

## 6. Single-coin backtest (next-bar-open execution)

Long-only: enter when `pred_return > threshold`, executed at the **next bar's open** (never the same bar the model observed). Fees + slippage applied. Outputs stats JSON, trades CSV, and equity CSV.

In [ ]:
!.venv-backtest/bin/python -m src.backtesting.vbt_adapter \
    --predictions "{BTC_PRED_CSV}" \
    --ohlc data/cache/lunarcrush_btc_4hour_full.csv \
    --outdir experiments/results/btc_4h_return/backtest \
    --label btc_hier_4h --freq 4h \
    --fees 0.001 --slippage 0.0005 --init-cash 10000

In [ ]:
# Show the full stats
import json
stats = json.load(open('experiments/results/btc_4h_return/backtest/btc_hier_4h_stats.json'))
for k, v in stats.items():
    print(f'{k:<28} {v}')

## 7. Equity & drawdown plots

In [ ]:
# Plots are pure matplotlib — render them in the main env
from src.backtesting.plots import plot_equity_and_drawdown
from IPython.display import Image, display

eq_png, dd_png = plot_equity_and_drawdown(
    'experiments/results/btc_4h_return/backtest/btc_hier_4h_equity.csv',
    outdir='experiments/results/btc_4h_return/backtest/figs',
    label='btc_hier_4h',
)
display(Image(filename=eq_png, width=800))
display(Image(filename=dd_png, width=800))

## 8. Multi-coin basket (pooled NAV)

Train ETH the same way, then run a **pooled** BTC+ETH portfolio with `cash_sharing=True` — equal-weight among active longs each bar. Requires `lunarcrush_eth_4hour_full.csv` in `data/cache/`.

In [ ]:
# Train ETH in return-mode (skip if you only want single-coin)
if os.path.exists('data/cache/lunarcrush_eth_4hour_full.csv'):
    make_return_config('configs/data/4hourly/btc_hier.yaml', 'eth',
                       'configs/data/4hourly/eth_hier_return.yaml',
                       'experiments/results/eth_4h_return')
    # reuse the same experiment config (variant + seed)
    eth_exp = exp_yaml.replace('btc_4h_return', 'eth_4h_return')
    open('configs/experiments/eth_4h_return.yaml', 'w').write(eth_exp)

    !python experiments/run_experiments.py \
        --base-config configs/data/4hourly/eth_hier_return.yaml \
        --experiment-config configs/experiments/eth_4h_return.yaml
else:
    print('ETH cache not found — upload lunarcrush_eth_4hour_full.csv to run the basket.')

In [ ]:
import glob
eth_csvs = glob.glob('experiments/results/eth_4h_return/*_predictions.csv')
if eth_csvs:
    ETH_PRED_CSV = eth_csvs[0]
    !.venv-backtest/bin/python -m src.backtesting.basket \
        --coin "btc:{BTC_PRED_CSV}:data/cache/lunarcrush_btc_4hour_full.csv" \
        --coin "eth:{ETH_PRED_CSV}:data/cache/lunarcrush_eth_4hour_full.csv" \
        --outdir experiments/results/basket_4h_return --label btc_eth_basket --freq 4h \
        --fees 0.001 --slippage 0.0005 --init-cash 100000
else:
    print('No ETH predictions — run the ETH training cell above first.')

In [ ]:
# Basket equity / drawdown plot (single pooled NAV)
basket_eq = 'experiments/results/basket_4h_return/btc_eth_basket_equity.csv'
if os.path.exists(basket_eq):
    from src.backtesting.plots import plot_equity_and_drawdown
    from IPython.display import Image, display
    eq_png, dd_png = plot_equity_and_drawdown(
        basket_eq, outdir='experiments/results/basket_4h_return/figs',
        label='btc_eth_basket',
    )
    display(Image(filename=eq_png, width=800))
    display(Image(filename=dd_png, width=800))

## 9. Save everything to Google Drive

In [ ]:
import shutil
from datetime import datetime

GDRIVE_OUTPUT_DIR = '/content/drive/MyDrive/thesis_results_returns_backtest'
os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)
run_dir = f"{GDRIVE_OUTPUT_DIR}/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(run_dir, exist_ok=True)

for sub in ['btc_4h_return', 'eth_4h_return', 'basket_4h_return']:
    src = f'experiments/results/{sub}'
    if os.path.exists(src):
        shutil.copytree(src, f'{run_dir}/{sub}', dirs_exist_ok=True)
        print('saved', sub)

print(f'\nAll results saved to: {run_dir}')
!ls -R {run_dir} | head -40

## Notes

- **`target_mode: return`** is the only change from the standard pipeline; `price` mode is the unchanged default.
- The legacy `test_metrics.sharpe_ratio` in the results JSON is the *toy* sign-of-(pred−prev) Sharpe. The **real, fee-aware** Sharpe lives in the backtest `stats.json`.
- Execution is **next-bar open**, never same-bar close, to avoid look-ahead optimism.
- For a stronger run, raise `num_epochs` (e.g. 100) and consider tuning `--threshold` in the backtest to trade less.
- Full reference: `docs/BACKTESTING.md` in the repo.